# 素体の2Dモデル用: see-through を Colab で動かす

novelai-userscripts-example の「素体 → 2Dモデル」で、絵を層(前髪・後ろ髪・顔・目・服・腕など最大 23 層、隠れた所まで描き足したもの)に
分けるためのリモートの GPU です([see-through](https://github.com/mitmul/see-through)、SIGGRAPH 2026)。

1. メニューの「ランタイム → ランタイムのタイプを変更」で **L4**(か A100)を選ぶ(bf16 が要るので T4 は不向き)
2. 上から順にセルを実行する(はじめは依存の導入とモデルの取得で 10〜15 分)
3. 最後のセルに出た 2 行を、アプリの `.env` に書いてバックエンドを起動し直す

終わったら「ランタイム → ランタイムを接続解除して削除」で止めてください(止めるまでコンピューティングユニットが減ります)。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. see-through を入れる

Colab の PyTorch をそのまま使い、see-through の依存だけを入れます(PyQt などの画面用のものは外します)。

In [ ]:
%cd /content
![ -d see-through ] || git clone -q --depth 1 https://github.com/mitmul/see-through
%cd /content/see-through
!grep -v -E "^(PyQt6|PyQt6-Qt6|qtpy|ipykernel|pytest|lpips|convnext_perceptual_loss)" requirements.txt > /content/req_colab.txt
!pip -q install -r /content/req_colab.txt
!pip -q install fastapi uvicorn python-multipart
!ln -sf common/assets assets
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

## 2. 試しに1枚分ける(モデルの取得を兼ねる)

LayerDiff 3D(SDXL)と Marigold の重みを取ってきます。付属の絵で1度通して、動くことを確かめます。

In [ ]:
%cd /content/see-through
!python inference/scripts/inference_psd.py --srcp assets/test_image.png --save_to_psd --save_dir /content/test_out --disable_progressbar
!find /content/test_out -name "*.psd" | head

## 3. 合言葉付きのサーバーを起動する

アプリから絵を受け取って PSD を返すサーバーです。Colab の中だけで待ち、合言葉(トークン)が無い呼び出しは断ります。

In [ ]:
import os, secrets, subprocess, time, urllib.request

BRANCH = "main"
urllib.request.urlretrieve(
    f"https://raw.githubusercontent.com/hirotoitpost/novelai-userscripts-example/{BRANCH}/scripts/see_through_server.py",
    "/content/see-through/see_through_server.py",
)
TOKEN = globals().get("TOKEN") or secrets.token_urlsafe(32)
server = subprocess.Popen(
    ["python", "see_through_server.py", "--port", "8190"],
    cwd="/content/see-through", env={**os.environ, "SEE_THROUGH_TOKEN": TOKEN},
    stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(60):
    try:
        req = urllib.request.Request("http://127.0.0.1:8190/health", headers={"Authorization": f"Bearer {TOKEN}"})
        print(urllib.request.urlopen(req, timeout=2).read().decode())
        break
    except Exception:
        time.sleep(2)

## 4. 外から使えるようにする(cloudflared)

URL はこのセルを実行するたびに変わります。

In [ ]:
import re, subprocess, threading, time, urllib.request

urllib.request.urlretrieve(
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "/content/cloudflared"
)
!chmod +x /content/cloudflared
tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8190"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
url = None
start = time.time()
while url is None and time.time() - start < 60:
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tunnel.stdout.readline())
    if match:
        url = match.group(0)
if url is None:
    tunnel.kill()
    raise RuntimeError("60 秒たっても cloudflared の URL が出ませんでした。このセルをもう一度実行してください。")
# 出力を読み続けないと cloudflared が詰まって止まるので、残りは捨て続ける
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()
print("アプリの .env に書く 2 行:")
print(f"SEE_THROUGH_BASE_URL={url}")
print(f"SEE_THROUGH_TOKEN={TOKEN}")

## 5. 使い終わったら

このノートブックを開いたままにすると、使っていなくてもユニットが減ります。終わったら
「ランタイム → ランタイムを接続解除して削除」を選んでください。ログは `/content/server.log` にあります。

In [ ]:
!tail -n 30 /content/server.log